In [1]:
!pip install transformers>=4.35.0
!pip install datasets>=2.14.0
!pip install evaluate>=0.4.0
!pip install rouge-score>=0.1.2
!pip install streamlit>=1.28.0
!pip install pandas numpy matplotlib seaborn
!pip install sentencepiece protobuf accelerate
!pip install scikit-learn

In [16]:
from datasets import load_dataset
from transformers import T5Tokenizer

ds = load_dataset("b-mc2/wikihow_lists")


ds = ds["train"].select_columns(["title", "result"]).shuffle(seed=42)

total = ds.shape[0]
train_size = int(total * 0.8)
test_size = int(total * 0.1)
val_size = total - train_size - test_size

train_ds = ds.select(range(0, train_size))
test_ds = ds.select(range(train_size, train_size + test_size))
val_ds = ds.select(range(train_size + test_size, total))

tokenizer = T5Tokenizer.from_pretrained('t5-small')

def preprocess_function(examples):
  inputs = ['summarize: ' + doc for doc in examples['result']]

  model_inputs = tokenizer(
      inputs,
      max_length = 512,
      truncation = True,
      padding = 'max_length',
  )

  labels = tokenizer(
      examples['title'],
      max_length = 64,
      truncation = True
  )

  model_inputs['labels'] = labels['input_ids']
  return model_inputs

train_dataset = train_ds.map(preprocess_function, batched=True, remove_columns=["title", "result"])
val_dataset = val_ds.map(preprocess_function, batched=True, remove_columns=["title", "result"])


Map:   0%|          | 0/9168 [00:00<?, ? examples/s]

Map:   0%|          | 0/1147 [00:00<?, ? examples/s]

In [18]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("t5-small")
# Training configuration
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_summarization_model",
    eval_strategy="steps",          # Updated from 'evaluation_strategy'
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=500,
    gradient_accumulation_steps=2,
    fp16=True,
    predict_with_generate=True,     # Required for seq2seq generation metrics (e.g., ROUGE)
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,             # Optional: keeps disk from filling up with old checkpoints
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [19]:
from transformers import Trainer, DataCollatorForSeq2Seq

# Data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)
# Start training
trainer.train()
# Save model
trainer.save_model()
tokenizer.save_pretrained("./t5_summarization_model")

Step,Training Loss,Validation Loss
500,9.280042,3.062655
1000,6.608091,2.870967
1500,6.303111,2.811685
2000,6.171416,2.776564
2500,6.100867,2.765723
2865,6.100867,2.762292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./t5_summarization_model/tokenizer_config.json',
 './t5_summarization_model/tokenizer.json')

In [23]:
import shutil

shutil.make_archive("model", "zip", '/content/t5_summarization_model/checkpoint-2865')

'/content/model.zip'

In [9]:
!pip install evaluate

In [12]:
!pip install -q rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [20]:
import torch
from evaluate import load
from tqdm import tqdm

rouge = load("rouge")

def evaluate_model(model, tokenizer, test_ds, batch_size=16):
    model.eval()
    device = model.device
    predictions = []
    references = []

    # Process in batches for much faster GPU evaluation
    for i in tqdm(range(0, len(test_ds), batch_size)):
        batch = test_ds[i : i + batch_size]
        inputs = [f"summarize: {doc}" for doc in batch["result"]]

        encoded = tokenizer(
            inputs,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_new_tokens=128,
                num_beams=4,
                length_penalty=0.6,
                early_stopping=True
            )

        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded_preds)
        references.extend(batch["title"])

    # Calculate ROUGE
    results = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    return results

In [21]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer

def load_model(model_path="./t5_model"):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()  # Disable dropout for deterministic inference

    return model, tokenizer, device

model, tokenizer, device = load_model('/content/t5_summarization_model/checkpoint-2865')

results = evaluate_model(model, tokenizer, test_ds)

print(results)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

100%|██████████| 72/72 [00:15<00:00,  4.64it/s]


{'rouge1': np.float64(0.37622385585110263), 'rouge2': np.float64(0.13881522283508752), 'rougeL': np.float64(0.3677129781051617), 'rougeLsum': np.float64(0.36786718097563686)}


In [22]:
import json

# Convert numpy floats to native Python floats and round for readability
clean_scores = {k: round(float(v), 4) for k, v in results.items()}

with open("rouge_results.json", "w") as f:
    json.dump(clean_scores, f, indent=4)

print("Saved ROUGE results:", clean_scores)

Saved ROUGE results: {'rouge1': 0.3762, 'rouge2': 0.1388, 'rougeL': 0.3677, 'rougeLsum': 0.3679}
